In [0]:
# -------------------------------------------------------------------
# 1. Read from Bronze Unity Catalog Table
# -------------------------------------------------------------------
catalog_name = "dbr_dev"
login_prefix = "joshuandegwa"
schema_name = f"{login_prefix}_bronze"
table_name = "opensky_flights_stream"

table_path = f"{catalog_name}.{schema_name}.{table_name}"

# Load data into a DataFrame for exploration
df = spark.read.table(table_path)
display(df)

In [0]:
%sql
select count(*) from dbr_dev.joshuandegwa_bronze.bronze_opensky_flights;

In [0]:
%sql
select * from dbr_dev.joshuandegwa_bronze.bronze_opensky_flights
order by `_api_snapshot_time` desc limit 10;

In [0]:
display(df)

In [0]:
%sql
SELECT 
    date_trunc('minute', meta_processed_at) AS processing_minute,
    COUNT(*) AS record_count
FROM dbr_dev.joshuandegwa_bronze.opensky_flights_stream
GROUP BY processing_minute
ORDER BY processing_minute DESC;

In [0]:
%sql
SELECT * FROM dbr_dev.joshuandegwa_bronze.opensky_flights_stream LIMIT 10;

In [0]:
import plotly.express as px

# 1. Altitude vs. Velocity Scatter Plot
fig_scatter = px.scatter(
    df,
    x="Velocity",
    y="Altitude",
    text="Callsign",
    color="Origin Country",
    title="Aircraft Altitude vs. Velocity",
    labels={
        "Velocity": "Velocity (m/s)",
        "Altitude": "Altitude (m)",
    },
)
fig_scatter.update_traces(textposition="top center")
fig_scatter.show()

# 2. Origin Country Frequency Bar Chart
country_counts = df["Origin Country"].value_counts().reset_index()
country_counts.columns = ["Origin Country", "Count"]

fig_bar = px.bar(
    country_counts,
    x="Origin Country",
    y="Count",
    color="Origin Country",
    title="Flight Distribution by Origin Country",
)
fig_bar.show()

In [0]:
%sql
DROP TABLE IF EXISTS workspace.joshuandegwa_gold.gold_origin_country_summary;

In [0]:
%sql
DROP SCHEMA IF EXISTS workspace.joshuandegwa_gold CASCADE;